# Assignment 4 - Fraud Detection (Prediction Notebook)
### PyTorch Version

**Instructions:**
1. Set `TEST_FILE` below to your test CSV path.
2. Ensure these four files are in the same folder as this notebook:
   - `fraud_mlp_state_dict.pt`
   - `fraud_scaler.joblib`
   - `fraud_encoder.joblib`
   - `fraud_feature_meta.joblib`
3. Click **Run All**.

In [ ]:
# ── USER CONFIGURATION ──────────────────────────────────────────────
TEST_FILE = "fraudTest.csv"   # <-- change to your test file path
THRESHOLD = 0.5               # classification threshold (0-1)
# ────────────────────────────────────────────────────────────────────

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

from sklearn.metrics import (
    classification_report, confusion_matrix,
    ConfusionMatrixDisplay, roc_auc_score
)

# Device
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"Using device: {device}")

## Step 1 — Define Model Architecture

Must match exactly what was used during training.

In [ ]:
class FraudMLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(64, 32),
            nn.ReLU(),

            nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.net(x)

## Step 2 — Load Saved Artifacts

In [ ]:
scaler       = joblib.load("fraud_scaler.joblib")
encoder      = joblib.load("fraud_encoder.joblib")
feature_meta = joblib.load("fraud_feature_meta.joblib")

numeric_cols     = feature_meta["numeric_cols"]
categorical_cols = feature_meta["categorical_cols"]

# Reconstruct input dimension from scaler + encoder
input_dim = len(numeric_cols) + len(encoder.get_feature_names_out(categorical_cols))
print(f"Input dimension: {input_dim}")

model = FraudMLP(input_dim).to(device)
model.load_state_dict(torch.load("fraud_mlp_state_dict.pt", map_location=device))
model.eval()
print("Model loaded successfully.")

## Step 3 — Feature Engineering & Preprocessing Function

Applies the exact same transformations used during training.

In [ ]:
def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2.0) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0) ** 2
    return R * 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))


def prepare_data(df_raw):
    """
    Feature-engineer and preprocess a raw DataFrame (same schema as fraudTrain.csv).
    Returns X (numpy float32 array) and y (array of true labels, or None).
    """
    df = df_raw.copy()
    if "Unnamed: 0" in df.columns:
        df = df.drop(columns=["Unnamed: 0"])

    # Location
    df["distance_km"] = haversine_km(
        df["lat"], df["long"], df["merch_lat"], df["merch_long"]
    )

    # Time
    df["trans_date_trans_time"] = pd.to_datetime(df["trans_date_trans_time"])
    df["dob"] = pd.to_datetime(df["dob"])
    df["hour"]         = df["trans_date_trans_time"].dt.hour
    df["day_of_week"]  = df["trans_date_trans_time"].dt.dayofweek
    df["month"]        = df["trans_date_trans_time"].dt.month
    df["customer_age"] = ((df["trans_date_trans_time"] - df["dob"]).dt.days / 365.25).astype(float)
    df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
    df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)
    df["dow_sin"]  = np.sin(2 * np.pi * df["day_of_week"] / 7)
    df["dow_cos"]  = np.cos(2 * np.pi * df["day_of_week"] / 7)

    # Separate target if present
    y = df["is_fraud"].values.astype(np.float32) if "is_fraud" in df.columns else None

    # Encode & scale using fitted artifacts
    X_cat = encoder.transform(df[categorical_cols])
    X_num = scaler.transform(df[numeric_cols])
    X = np.hstack([X_num, X_cat]).astype(np.float32)

    return X, y

## Step 4 — Load & Prepare Test Data

In [ ]:
print(f"Loading: {TEST_FILE}")
df_test = pd.read_csv(TEST_FILE)
print(f"Rows: {len(df_test):,}   Columns: {df_test.shape[1]}")

X_test, y_test = prepare_data(df_test)
print(f"Feature matrix: {X_test.shape}")

## Step 5 — Make Predictions

In [ ]:
test_ds     = TensorDataset(torch.tensor(X_test, dtype=torch.float32))
test_loader = DataLoader(test_ds, batch_size=2048, shuffle=False)

all_probs = []
with torch.no_grad():
    for (X_batch,) in test_loader:
        X_batch = X_batch.to(device)
        probs = torch.sigmoid(model(X_batch)).cpu().numpy().flatten()
        all_probs.extend(probs)

all_probs = np.array(all_probs)
y_pred    = (all_probs >= THRESHOLD).astype(int)

print(f"Predicted fraud : {y_pred.sum():,}")
print(f"Predicted legit : {(y_pred == 0).sum():,}")

## Step 6 — Confusion Matrix & Classification Report

In [ ]:
if y_test is not None:
    print(f"ROC-AUC: {roc_auc_score(y_test, all_probs):.4f}")
    print()
    print(classification_report(y_test, y_pred, target_names=["Legit", "Fraud"]))

    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Legit", "Fraud"])
    fig, ax = plt.subplots(figsize=(6, 5))
    disp.plot(cmap="Blues", ax=ax)
    ax.set_title("Confusion Matrix — Test Set")
    plt.tight_layout()
    plt.show()

    tn, fp, fn, tp = cm.ravel()
    print(f"True Negatives  (Legit correctly identified) : {tn:>8,}")
    print(f"False Positives (Legit flagged as Fraud)     : {fp:>8,}")
    print(f"False Negatives (Fraud missed)               : {fn:>8,}")
    print(f"True Positives  (Fraud correctly caught)     : {tp:>8,}")
else:
    print("No 'is_fraud' column found — skipping confusion matrix.")
    print("Predictions stored in y_pred (0=Legit, 1=Fraud).")

## Step 7 — Export Predictions (Optional)

In [ ]:
output = pd.DataFrame({
    "predicted_label":   y_pred,
    "fraud_probability": all_probs
})
if y_test is not None:
    output["true_label"] = y_test.astype(int)

output.to_csv("predictions.csv", index=False)
print("Predictions saved to predictions.csv")
output.head(10)